
# SVG → Centerline experiments

Цель: превратить залитую SVG-фигуру в компактную центральную линию.

Основной вариант здесь **полностью векторный**:

`SVG → svgelements → adaptive path sampling → Shapely Polygon → pygeoops.centerline`

Растеризация оставлена только как независимый baseline для сравнения.


In [ ]:

#@title 1. Install
!pip -q install svgelements shapely pygeoops matplotlib cairosvg scikit-image pillow


In [ ]:

#@title 2. Imports
from pathlib import Path
import io, math, time
import numpy as np
import matplotlib.pyplot as plt

from svgelements import SVG, Path as SvgPath, Shape, Move, Close
from shapely.geometry import Polygon, MultiPolygon, LineString, MultiLineString
from shapely.ops import unary_union, linemerge
import pygeoops

from google.colab import files
from IPython.display import display, SVG as DisplaySVG

WORK = Path("/content/centerline")
WORK.mkdir(exist_ok=True)


In [ ]:

#@title 3. Upload SVG
uploaded = files.upload()
name = next(n for n in uploaded if n.lower().endswith(".svg"))
svg_path = WORK / name
svg_path.write_bytes(uploaded[name])

display(DisplaySVG(filename=str(svg_path)))



## A. Vector SVG → Polygon

`svgelements` разбирает настоящий SVG: path-команды, Bézier, arc, transforms,
basic shapes и `<use>`.

Shapely не хранит кривые Bézier напрямую, поэтому кривую надо представить
достаточно точной полилинией. Это **не rasterization**: точки берутся прямо
с математической SVG-кривой.


In [ ]:

#@title 4. Parse filled SVG paths

svg = SVG.parse(str(svg_path), reify=True)

paths = []
for element in svg.elements():
    if isinstance(element, SvgPath):
        p = element
    elif isinstance(element, Shape):
        p = SvgPath(element)
        p.reify()
    else:
        continue

    if len(p) and str(getattr(element, "fill", "black")).lower() != "none":
        paths.append(p)

print("Filled vector paths:", len(paths))
for i, p in enumerate(paths):
    print(i, "segments:", len(p), "bbox:", p.bbox())


In [ ]:

#@title 5. Adaptive-ish vector sampling

# Максимальная длина прямого кусочка полилинии в SVG units.
# Для GClef с width≈15 значение 0.02 уже очень плотное.
MAX_STEP = 0.02  #@param {type:"number"}

def sample_segment(seg, max_step):
    length = max(float(seg.length(error=1e-6)), 1e-9)
    n = max(2, int(math.ceil(length / max_step)) + 1)
    return [(float(seg.point(t).x), float(seg.point(t).y))
            for t in np.linspace(0.0, 1.0, n)]

def path_subpaths_to_rings(path, max_step):
    rings, current = [], []

    for seg in path:
        if isinstance(seg, Move):
            if len(current) >= 3:
                rings.append(current)
            current = [(float(seg.end.x), float(seg.end.y))]
            continue

        pts = sample_segment(seg, max_step)

        if not current:
            current.append(pts[0])

        current.extend(pts[1:])

        if isinstance(seg, Close):
            if len(current) >= 3:
                if current[0] != current[-1]:
                    current.append(current[0])
                rings.append(current)
            current = []

    if len(current) >= 3:
        if current[0] != current[-1]:
            current.append(current[0])
        rings.append(current)

    return rings

all_rings = []
for path in paths:
    all_rings.extend(path_subpaths_to_rings(path, MAX_STEP))

print("Closed rings:", len(all_rings))
print("Points:", [len(r) for r in all_rings])


In [ ]:

#@title 6. Rings → filled Shapely geometry

# Для compound SVG path с nonzero fill rule удобно собрать кольца
# через symmetric_difference. Для типовых глифов это корректно сохраняет
# внешние области и отверстия независимо от порядка subpath-ов.

geometry = None
for ring in all_rings:
    poly = Polygon(ring)
    if not poly.is_valid:
        poly = poly.buffer(0)
    if poly.is_empty:
        continue
    geometry = poly if geometry is None else geometry.symmetric_difference(poly)

if geometry is None:
    raise ValueError("No filled polygon produced")

geometry = geometry.buffer(0)

print("Geometry:", geometry.geom_type)
print("Area:", geometry.area)
print("Bounds:", geometry.bounds)

fig, ax = plt.subplots(figsize=(7, 12))
polys = [geometry] if geometry.geom_type == "Polygon" else list(geometry.geoms)
for p in polys:
    x, y = p.exterior.xy
    ax.plot(x, y)
    for hole in p.interiors:
        x, y = hole.xy
        ax.plot(x, y)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_title("Vector-derived polygon")
plt.show()



> Примечание: для общего SVG позже стоит уважать `fill-rule="nonzero|evenodd"`
> буквально. Для нашего текущего глифа compound path простой; эта ячейка
> намеренно минимальна для эксперимента.



## B. Vector geometry cleanup before centerline

Нижний хвост у некоторых глифов может содержать маленькие петли/перекрытия,
которые являются частью исходной заливки, но нам не нужны как отдельная
структура центральной линии.

Проверяем **векторную морфологию Shapely**, без rasterization:

- `opening`: `buffer(-ε).buffer(+ε)` — убирает тонкие выступы/перемычки;
- `closing`: `buffer(+ε).buffer(-ε)` — закрывает узкие щели.

Сначала делаем дешёвый sweep только по polygon. `centerline` считаем один раз
для выбранного варианта, чтобы не умножать время выполнения.


In [ ]:

#@title 7. Cleanup epsilon sweep

EPS_VALUES = [0.0, 0.02, 0.05, 0.10, 0.20, 0.30]
CLEAN_MODE = "opening"  #@param ["opening", "closing"]

def clean_geometry(g, epsilon, mode):
    if epsilon <= 0:
        return g

    if mode == "opening":
        result = g.buffer(-epsilon).buffer(epsilon)
    elif mode == "closing":
        result = g.buffer(epsilon).buffer(-epsilon)
    else:
        raise ValueError(mode)

    return result.buffer(0)

def iter_polygons(g):
    if g.geom_type == "Polygon":
        return [g]
    if g.geom_type == "MultiPolygon":
        return list(g.geoms)
    return [x for x in getattr(g, "geoms", []) if x.geom_type == "Polygon"]

def plot_polygon(ax, g, title):
    for p in iter_polygons(g):
        x, y = p.exterior.xy
        ax.plot(x, y)
        for hole in p.interiors:
            x, y = hole.xy
            ax.plot(x, y)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_title(title)
    ax.axis("off")

fig, axes = plt.subplots(
    1, len(EPS_VALUES),
    figsize=(3.2 * len(EPS_VALUES), 10),
    squeeze=False
)

cleanup_candidates = {}

for ax, eps in zip(axes[0], EPS_VALUES):
    g = clean_geometry(geometry, eps, CLEAN_MODE)
    cleanup_candidates[eps] = g

    delta_area = g.area - geometry.area
    plot_polygon(
        ax, g,
        f"ε={eps:g}\nΔarea={delta_area:+.4f}"
    )

plt.tight_layout()
plt.show()


In [ ]:

#@title 8. Choose cleanup

# После sweep выставляем значение глазами.
# 0.0 = вообще не менять исходную геометрию.
CLEAN_EPSILON = 0.0  #@param {type:"number"}

cleaned_geometry = clean_geometry(geometry, CLEAN_EPSILON, CLEAN_MODE)

print("cleanup mode:", CLEAN_MODE)
print("epsilon:", CLEAN_EPSILON)
print("area before:", geometry.area)
print("area after: ", cleaned_geometry.area)
print("Δarea:", cleaned_geometry.area - geometry.area)

fig, axes = plt.subplots(1, 2, figsize=(10, 10))
plot_polygon(axes[0], geometry, "Original vector polygon")
plot_polygon(axes[1], cleaned_geometry, f"Cleaned: {CLEAN_MODE}, ε={CLEAN_EPSILON:g}")
plt.tight_layout()
plt.show()


In [ ]:

#@title 9. pygeoops centerline

DENSIFY_DISTANCE = -1.0    #@param {type:"number"}
MIN_BRANCH_LENGTH = -1.0   #@param {type:"number"}
SIMPLIFY = -0.25           #@param {type:"number"}

t0 = time.perf_counter()

centerline = pygeoops.centerline(
    cleaned_geometry,
    densify_distance=DENSIFY_DISTANCE,
    min_branch_length=MIN_BRANCH_LENGTH,
    simplifytolerance=SIMPLIFY,
)

elapsed = time.perf_counter() - t0

print(centerline.geom_type, "length =", centerline.length)
print(f"centerline time: {elapsed:.3f} s")

fig, ax = plt.subplots(figsize=(7, 12))
plot_polygon(ax, cleaned_geometry, "pygeoops centerline from cleaned SVG vectors")

lines = (
    [centerline]
    if centerline.geom_type == "LineString"
    else list(centerline.geoms)
)

for i, line in enumerate(lines):
    x, y = line.xy
    ax.plot(x, y, linewidth=1.6, label=str(i))

ax.legend(loc="upper right", fontsize=8)
plt.show()



## C. LineString diagnostics and free merging

`pygeoops.centerline` возвращает граф как набор `LineString`.
Разные цвета на графике — именно разные компоненты.

`linemerge()` бесплатно склеивает цепочки там, где топология однозначна
(в вершине степень 2). Через настоящие junction'ы он специально не идёт —
их будем распутывать отдельной эвристикой.


In [ ]:

#@title 10. Segment statistics before merge

def as_lines(g):
    if g.geom_type == "LineString":
        return [g]
    if g.geom_type == "MultiLineString":
        return list(g.geoms)
    return [x for x in getattr(g, "geoms", []) if x.geom_type == "LineString"]

raw_lines = as_lines(centerline)

print("Raw LineStrings:", len(raw_lines))
print()

for i, line in enumerate(raw_lines):
    coords = list(line.coords)
    start = tuple(round(v, 3) for v in coords[0])
    end = tuple(round(v, 3) for v in coords[-1])

    print(
        f"{i:2d}: length={line.length:8.3f} "
        f"points={len(coords):3d} "
        f"{start} -> {end}"
    )


In [ ]:

#@title 11. linemerge

merged_centerline = linemerge(centerline)
merged_lines = as_lines(merged_centerline)

print("Before:", len(raw_lines), "LineStrings")
print("After: ", len(merged_lines), "LineStrings")
print("Merged automatically:", len(raw_lines) - len(merged_lines))

fig, axes = plt.subplots(1, 2, figsize=(12, 12))

plot_polygon(axes[0], cleaned_geometry, f"Before linemerge: {len(raw_lines)}")
for i, line in enumerate(raw_lines):
    x, y = line.xy
    axes[0].plot(x, y, linewidth=1.6)

plot_polygon(axes[1], cleaned_geometry, f"After linemerge: {len(merged_lines)}")
for i, line in enumerate(merged_lines):
    x, y = line.xy
    axes[1].plot(x, y, linewidth=1.8, label=str(i))

axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:

#@title 12. Merged LineString statistics

for i, line in enumerate(merged_lines):
    coords = list(line.coords)
    start = tuple(round(v, 3) for v in coords[0])
    end = tuple(round(v, 3) for v in coords[-1])

    print(
        f"{i:2d}: length={line.length:8.3f} "
        f"points={len(coords):3d} "
        f"{start} -> {end}"
    )



## D. Raster medial axis — только baseline

Этот вариант оставляем только для независимого сравнения и для получения
distance-to-boundary (локальной полуширины ленты). В основном векторном
pipeline он не участвует.


In [ ]:

#@title 13. Optional raster baseline
import cairosvg
from PIL import Image
from skimage.morphology import medial_axis

SCALE = 12  #@param {type:"integer"}

png = cairosvg.svg2png(
    bytestring=svg_path.read_bytes(),
    scale=SCALE,
    background_color="white"
)
gray = np.array(Image.open(io.BytesIO(png)).convert("L"))
mask = gray < 245

skeleton, distance = medial_axis(mask, return_distance=True)
ys, xs = np.nonzero(skeleton)

plt.figure(figsize=(7, 12))
plt.imshow(mask, cmap="gray")
plt.scatter(xs, ys, s=.25)
plt.axis("off")
plt.title("Raster medial axis baseline")
plt.show()

r = distance[skeleton]
print("skeleton pixels:", len(r))
print("width median:", float(np.median(r) * 2))



## Следующий шаг

Теперь у нас отдельно видны:

1. артефакты исходной vector geometry;
2. эффект мягкой cleanup-операции до centerline;
3. сколько сегментов создаёт `pygeoops`;
4. сколько из них `linemerge()` умеет склеить без эвристик.

Если после `linemerge` остаются junction'ы, следующий этап — построить граф
endpoint/junction и попарно соединять рёбра по касательной, кривизне и,
при необходимости, локальной ширине ленты.
